# Can a model identify room contents and how much space they take?

Two methods, run side by side on the same input, scored against hand measurements.

| | Method | How volume is obtained |
|---|---|---|
| **Arm A** | Classify → look up | Decide *"3-seater sofa"*, read packed volume off `cube_table.json` |
| **Arm B** | Measure geometrically | Metric depth → 3D extent per object → compute volume |

Both shipping products in this market use **Arm A** and neither measures objects.
This notebook checks that on our own data instead of taking it on trust.

**Run order:** stages are sequential but each is independently re-runnable. Heavy model
cells (2, 3) load once and cache. Arm A (stage 6) needs `ANTHROPIC_API_KEY`; if it is
missing the notebook skips it and still runs Arm B.

> **The numbers mean nothing without `ground_truth.csv`.** See gap **A1** in `../GAPS.md`.
> Copy `ground_truth_template.csv`, fill it in for the rooms you shoot, and stage 9
> becomes an experiment instead of a demo.

## Stage 0 · Configuration

In [ ]:
from pathlib import Path
import json, os, sys

# resolve whether we're running from repo root or from poc/
ROOT = Path.cwd() if (Path.cwd() / "cube_table.json").exists() else Path.cwd() / "poc"
assert (ROOT / "cube_table.json").exists(), f"cube_table.json not found relative to {Path.cwd()}"
IN_DIR, OUT_DIR = ROOT / "data" / "input", ROOT / "data" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = dict(
    # ---- stage 1 ingest ----
    frame_stride_s   = 1.0,      # sample one frame per N seconds of video
    max_frames       = 16,
    blur_min         = 60.0,     # variance-of-Laplacian floor; below = too blurry to use
    bright_range     = (35, 225),
    resize_long_edge = 1024,

    # ---- stage 2 detection ----
    det_model  = "IDEA-Research/grounding-dino-base",
    det_box_th = 0.30,
    det_txt_th = 0.25,

    # ---- stage 3 depth ----  (MIT licence — see gap B2)
    depth_model = "Ruicheng/moge-2-vitl",   # smaller: moge-2-vitb-normal, moge-2-vits-normal

    # ---- stage 4 scale anchor ----  THIS IS THE ABLATION SWITCH (gap A5)
    use_scale_anchor = True,
    door_height_m    = 1.981,    # UK internal door leaf, the cheap metre-stick
    manual_scale     = None,     # set a float to force a correction factor instead

    # ---- stage 6 Arm A ----
    vlm_model      = "claude-sonnet-5",
    vlm_max_frames = 6,          # cost control; ~$0.02 per frame

    # ---- stage 7 dedup ----  (gap B1)
    dedup_rule = "max",          # "max" = safer against occlusion, "median" = safer against false positives
)

SCORE_ROOM = None            # stage 9: room_id to score, or None for the first one

CUBE  = json.loads((ROOT / "cube_table.json").read_text())
VOCAB = json.loads((ROOT / "detect_vocab.json").read_text())
CLASSES = CUBE["classes"]
CLASS_NAMES = sorted(CLASSES.keys())

print(f"root         {ROOT}")
print(f"cube classes {len(CLASS_NAMES)}   ({CUBE['_meta']['status'][:44]}...)")
print(f"det prompts  {len(VOCAB)}")
print(f"inputs       {sorted(p.name for p in IN_DIR.iterdir() if p.name != '.gitkeep')}")

In [ ]:
import numpy as np, cv2, torch, pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

def pick_device():
    if torch.cuda.is_available():          return torch.device("cuda")
    if torch.backends.mps.is_available():  return torch.device("mps")
    return torch.device("cpu")

DEVICE = pick_device()
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"torch {torch.__version__} on {DEVICE}")
print(f"ANTHROPIC_API_KEY {'set — Arm A enabled' if HAS_KEY else 'MISSING — Arm A will be skipped'}")
if DEVICE.type == "cpu":
    print("\n! CPU only. MoGe-2 vitl will be slow (~30s/frame). Consider moge-2-vits-normal.")

## Stage 1 · Ingest — video or image to sharp keyframes

A one-minute room video is ~1,800 frames, most of them blurred, redundant, or pointed at
the floor. Sample at a fixed interval, then gate on sharpness and brightness (gap **B5**).

The gate is deliberately visible: if it is throwing away most of your footage, the capture
instruction needs fixing, not the threshold.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
VID_EXT = {".mp4", ".mov", ".m4v", ".avi", ".mkv"}

def resize_long(img, long_edge):
    h, w = img.shape[:2]
    s = long_edge / max(h, w)
    return cv2.resize(img, (int(round(w*s)), int(round(h*s))), interpolation=cv2.INTER_AREA) if s < 1 else img

def sharpness(gray):
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def load_keyframes(path, cfg):
    """Return (all_sampled, kept) frame dicts."""
    raw = []
    if path.suffix.lower() in IMG_EXT:
        img = cv2.imread(str(path))
        if img is None: raise IOError(f"could not read {path}")
        raw = [(0.0, img)]
    elif path.suffix.lower() in VID_EXT:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        stride = max(1, int(round(fps * cfg["frame_stride_s"])))
        i = 0
        while True:
            ok, fr = cap.read()
            if not ok: break
            if i % stride == 0: raw.append((i / fps, fr))
            i += 1
        cap.release()
    else:
        raise ValueError(f"unsupported file type: {path.suffix}")

    out = []
    for t, fr in raw:
        fr = resize_long(fr, cfg["resize_long_edge"])
        g  = cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY)
        s, b = sharpness(g), float(g.mean())
        out.append(dict(t=t, img=fr, sharp=s, bright=b,
                        ok=(s >= cfg["blur_min"] and cfg["bright_range"][0] <= b <= cfg["bright_range"][1])))
    kept = [f for f in out if f["ok"]][: cfg["max_frames"]]
    return out, kept

inputs = sorted(p for p in IN_DIR.iterdir() if p.suffix.lower() in (IMG_EXT | VID_EXT))
if not inputs:
    raise SystemExit(f"No input files. Drop a room photo or video into {IN_DIR}")

TARGET = inputs[0]        # change index to process a different file
sampled, frames = load_keyframes(TARGET, CFG)
print(f"{TARGET.name}: sampled {len(sampled)}, kept {len(frames)}")
if sampled:
    print(f"  sharpness range {min(f['sharp'] for f in sampled):.0f} – {max(f['sharp'] for f in sampled):.0f} (floor {CFG['blur_min']})")
    print(f"  rejected {sum(1 for f in sampled if not f['ok'])}")

In [ ]:
n = min(len(frames), 8)
if n:
    fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
    for ax in axes.ravel(): ax.axis("off")
    for ax, f in zip(axes.ravel(), frames[:n]):
        ax.imshow(cv2.cvtColor(f["img"], cv2.COLOR_BGR2RGB))
        ax.set_title(f"t={f['t']:.1f}s  sharp={f['sharp']:.0f}", fontsize=9)
    plt.suptitle(f"Stage 1 — keyframes kept from {TARGET.name}", fontsize=11)
    plt.tight_layout(); plt.show()

## Stage 2 · Detection — open-vocabulary boxes

Grounding DINO with our furniture vocabulary. `door` is in the prompt but excluded from
inventory — it is there purely as the scale anchor for stage 4.

Note we ask the **detector** for boxes and counts, not the language model. Measured VLM
counting accuracy is around 0.53 and biased toward under-counting, which is the dangerous
direction (see gap notes). The detector counts; the VLM names.

In [ ]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from PIL import Image

_det = {}
def det_load():
    if "m" not in _det:
        print(f"loading {CFG['det_model']} ...")
        _det["p"] = AutoProcessor.from_pretrained(CFG["det_model"])
        _det["m"] = AutoModelForZeroShotObjectDetection.from_pretrained(CFG["det_model"]).to(DEVICE).eval()
    return _det["p"], _det["m"]

# Grounding DINO wants lowercase phrases separated by ". " and a trailing "."
DET_PROMPT = ". ".join(VOCAB.keys()).lower() + "."

def detect(img_bgr):
    proc, model = det_load()
    pil = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    inp = proc(images=pil, text=DET_PROMPT, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model(**inp)
    kw = dict(input_ids=inp["input_ids"], target_sizes=[pil.size[::-1]],
              text_threshold=CFG["det_txt_th"])
    try:      # transformers renamed box_threshold -> threshold around 4.51
        res = proc.post_process_grounded_object_detection(out, threshold=CFG["det_box_th"], **kw)[0]
    except TypeError:
        res = proc.post_process_grounded_object_detection(out, box_threshold=CFG["det_box_th"], **kw)[0]
    labels = res.get("labels") or res.get("text_labels")
    return [dict(label=str(l).strip().lower(), score=float(s), box=[float(v) for v in b])
            for l, s, b in zip(labels, res["scores"], res["boxes"])]

for f in frames:
    f["dets"] = detect(f["img"])

tot = sum(len(f["dets"]) for f in frames)
print(f"{tot} detections across {len(frames)} frames")
hist = defaultdict(int)
for f in frames:
    for d in f["dets"]: hist[d["label"]] += 1
for k, v in sorted(hist.items(), key=lambda x: -x[1]):
    print(f"  {k:22} {v}")

In [ ]:
def draw(img, dets):
    v = img.copy()
    for d in dets:
        x0, y0, x1, y1 = [int(z) for z in d["box"]]
        c = (60, 200, 190) if "door" in d["label"] else (40, 120, 240)
        cv2.rectangle(v, (x0, y0), (x1, y1), c, 2)
        cv2.putText(v, f"{d['label']} {d['score']:.2f}", (x0, max(14, y0 - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1, cv2.LINE_AA)
    return v

n = min(len(frames), 4)
if n:
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 4.2))
    axes = np.atleast_1d(axes)
    for ax, f in zip(axes, frames[:n]):
        ax.imshow(cv2.cvtColor(draw(f["img"], f["dets"]), cv2.COLOR_BGR2RGB))
        ax.set_title(f"t={f['t']:.1f}s · {len(f['dets'])} objects", fontsize=9); ax.axis("off")
    plt.suptitle("Stage 2 — detections (teal = door, used as scale anchor)", fontsize=11)
    plt.tight_layout(); plt.show()

## Stage 3 · Metric depth — MoGe-2

Returns a **metric point map** in metres: `points[y, x] = (X, Y, Z)` in camera coordinates,
plus predicted intrinsics and a validity mask. MIT licensed, so it is safe to ship — unlike
UniDepthV2, which posts better indoor numbers and is non-commercial (gap **B2**).

Published metric point-map error is **8.19%**. Because volume goes as the cube of length,
that is roughly **26% on volume** — and it is a *single multiplier on the whole room*, not
noise that averages out. Stage 4 exists to attack exactly that.

In [ ]:
_dep = {}
def depth_load():
    if "m" not in _dep:
        from moge.model.v2 import MoGeModel
        print(f"loading {CFG['depth_model']} ...")
        _dep["m"] = MoGeModel.from_pretrained(CFG["depth_model"]).to(DEVICE).eval()
    return _dep["m"]

def infer_depth(img_bgr):
    model = depth_load()
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    t = torch.tensor(rgb / 255.0, dtype=torch.float32, device=DEVICE).permute(2, 0, 1)
    with torch.no_grad():
        o = model.infer(t)
    return {k: (v.detach().cpu().numpy() if torch.is_tensor(v) else v) for k, v in o.items()}

for i, f in enumerate(frames):
    f["depth"] = infer_depth(f["img"])
    print(f"  frame {i+1}/{len(frames)} done", end="\r")

d0 = frames[0]["depth"]
print("\noutput keys:", list(d0.keys()))
pts, msk = d0["points"], d0["mask"].astype(bool)
print(f"points {pts.shape}  valid {msk.mean()*100:.0f}%")
for ax, nm in zip(range(3), "XYZ"):
    v = pts[..., ax][msk]
    print(f"  {nm}: {v.min():+.2f} .. {v.max():+.2f} m   (span {v.max()-v.min():.2f} m)")
print("\n! Verify the axis convention on your first real frame: we assume Y is the vertical")
print("  axis for door-height anchoring in stage 4. If the Y span looks like room width,")
print("  swap VERT_AXIS below.")
VERT_AXIS = 1

In [ ]:
f = frames[0]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].imshow(cv2.cvtColor(f["img"], cv2.COLOR_BGR2RGB)); ax[0].set_title("frame"); ax[0].axis("off")
dm = f["depth"]["depth"].copy()
dm[~f["depth"]["mask"].astype(bool)] = np.nan
im = ax[1].imshow(dm, cmap="viridis")
ax[1].set_title("metric depth (m)"); ax[1].axis("off")
plt.colorbar(im, ax=ax[1], fraction=0.035, label="metres")
plt.suptitle("Stage 3 — MoGe-2 metric depth", fontsize=11)
plt.tight_layout(); plt.show()

## Stage 4 · Scale anchor — the ablation switch

**This is the most important cell in the notebook.** Gap **A5**.

A UK internal door leaf is 1981 mm. Detect one, measure its height in the point map, and
the ratio is a correction factor for the entire room. One detection kills the error term
that averaging cannot touch.

Run the whole notebook twice — `use_scale_anchor` True then False — and compare stage 9.
**That delta is the single most valuable number this POC produces.**

In [ ]:
def door_scale(dets, points, mask, vert=VERT_AXIS):
    """Correction factor from a detected door, or None."""
    doors = [d for d in dets if "door" in d["label"]]
    if not doors: return None, "no door detected"
    d = max(doors, key=lambda x: x["score"])
    x0, y0, x1, y1 = [int(v) for v in d["box"]]
    sub, m = points[y0:y1, x0:x1], mask[y0:y1, x0:x1].astype(bool)
    if m.sum() < 200: return None, "door region has too few valid depth points"
    v = sub[..., vert][m]
    measured = float(np.percentile(v, 98) - np.percentile(v, 2))
    if measured < 0.3: return None, f"implausible door height {measured:.2f} m"
    return CFG["door_height_m"] / measured, f"door measured {measured:.3f} m (score {d['score']:.2f})"

factors = []
for f in frames:
    fac, why = door_scale(f["dets"], f["depth"]["points"], f["depth"]["mask"])
    f["scale_factor"], f["scale_why"] = fac, why
    if fac: factors.append(fac)

if CFG["manual_scale"] is not None:
    SCALE = float(CFG["manual_scale"]); src = "manual override"
elif CFG["use_scale_anchor"] and factors:
    SCALE = float(np.median(factors)); src = f"door anchor, median of {len(factors)} frame(s)"
else:
    SCALE = 1.0
    src = "NONE — raw model scale" + ("" if CFG["use_scale_anchor"] else " (anchor disabled)")

print(f"scale factor  {SCALE:.4f}   [{src}]")
if factors:
    print(f"  per-frame: {', '.join(f'{x:.3f}' for x in factors)}   spread {np.ptp(factors):.3f}")
    print(f"  -> implies raw depth was {abs(1-SCALE)*100:.1f}% {'small' if SCALE>1 else 'large'};"
          f" ~{abs(1-SCALE**3)*100:.0f}% on volume if left uncorrected")
else:
    for f in frames[:3]: print(f"  {f['scale_why']}")
    print("  ! No anchor. Volumes below carry the model's raw scale error (~26% expected).")

## Stage 5 · Arm B — measure each object geometrically

Camera-axis extents are wrong for anything not square to the lens, so take the height from
the vertical axis and run **PCA on the horizontal footprint** for width and depth. Robust
percentiles (2–98) reject depth outliers at object edges.

Then map the measured dimensions onto the nearest size class, which lets Arm B be scored
both as a *measurement* and as a *classifier*.

In [ ]:
def extent(points, mask, box, scale=1.0, vert=VERT_AXIS, lo=2, hi=98):
    x0, y0, x1, y1 = [int(v) for v in box]
    sub, m = points[y0:y1, x0:x1], mask[y0:y1, x0:x1].astype(bool)
    if m.sum() < 100: return None
    p = sub[m] * scale
    h = float(np.percentile(p[:, vert], hi) - np.percentile(p[:, vert], lo))
    horiz = np.delete(p, vert, axis=1)              # the two non-vertical axes
    horiz = horiz - horiz.mean(0)
    try:                                            # PCA -> oriented footprint
        _, _, vt = np.linalg.svd(horiz, full_matrices=False)
        proj = horiz @ vt.T
    except np.linalg.LinAlgError:
        proj = horiz
    w = float(np.percentile(proj[:, 0], hi) - np.percentile(proj[:, 0], lo))
    d = float(np.percentile(proj[:, 1], hi) - np.percentile(proj[:, 1], lo))
    w, d = max(w, d), min(w, d)
    return dict(w=abs(w), d=abs(d), h=abs(h), bbox_m3=abs(w*d*h), n_pts=int(m.sum()))

def nearest_class(dims, allowed=None):
    """Map measured w/d/h to the closest size class by sorted-dimension distance."""
    tgt = np.sort([dims["w"], dims["d"], dims["h"]])[::-1]
    best, bd = None, 1e9
    for name in (allowed or CLASS_NAMES):
        c = CLASSES.get(name)
        if not c: continue
        cand = np.sort(c["typical_dims_m"])[::-1]
        dist = float(np.linalg.norm(tgt - cand) / max(np.linalg.norm(cand), 1e-6))
        if dist < bd: best, bd = name, dist
    return best, bd

armB = []
for f in frames:
    for d in f["dets"]:
        if "door" in d["label"]: continue
        e = extent(f["depth"]["points"], f["depth"]["mask"], d["box"], scale=SCALE)
        if not e: continue
        allowed = VOCAB.get(d["label"]) or None
        cls, dist = nearest_class(e, allowed)
        armB.append(dict(t=f["t"], det_label=d["label"], score=d["score"], **e,
                         mapped_class=cls, map_dist=round(dist, 3),
                         cube_m3=CLASSES[cls]["cube_m3"] if cls else np.nan))

B = pd.DataFrame(armB)
if len(B):
    print(f"{len(B)} measured objects across {len(frames)} frames\n")
    print(B[["t","det_label","w","d","h","bbox_m3","mapped_class","map_dist"]]
          .round(3).head(20).to_string(index=False))
else:
    print("No measurable objects — check stage 2 detections and stage 3 mask coverage.")

## Stage 6 · Arm A — classify into size classes

The insight worth stealing from the incumbents: they distinguish *"queen mattress vs king
mattress"* and *"two-seater vs sectional"*. They are not measuring those — they are
**classifying into a size category, and the category carries the volume**. A discrete choice
between 8 sofa classes is a far more tractable problem than regressing three continuous
numbers.

Structured output is forced via a tool schema whose `enum` is our cube table, so the model
cannot invent a class we have no volume for.

In [ ]:
import base64

ARM_A_PROMPT = """You are cataloguing a room for a household removal survey.

List every MOVABLE item you can see. For each one pick the single closest size_class from
the allowed list — the class carries the packed volume, so choosing between e.g.
sofa_2_seat / sofa_3_seat / sofa_sectional matters as much as recognising it is a sofa.

Rules:
- Count only what is visible IN THIS IMAGE. Do not infer items you cannot see.
- Do not count fitted or structural things: fitted kitchen units, built-in wardrobes,
  radiators, doors, windows, flooring, light fittings.
- If an item is partly hidden, still count it once and say so in reasoning.
- If unsure between two size classes, pick the smaller and lower your confidence.
- confidence is 0.0-1.0 for the size_class choice, not for the item existing."""

def classify_frame(img_bgr, model=None):
    from anthropic import Anthropic
    client = Anthropic()
    ok, buf = cv2.imencode(".jpg", img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    b64 = base64.b64encode(buf.tobytes()).decode()
    tool = {
        "name": "record_inventory",
        "description": "Record every movable household item visible in this room image.",
        "input_schema": {
            "type": "object",
            "properties": {
                "room_type": {"type": "string", "description": "bedroom, living, kitchen, other"},
                "items": {"type": "array", "items": {
                    "type": "object",
                    "properties": {
                        "size_class": {"type": "string", "enum": CLASS_NAMES},
                        "count": {"type": "integer", "minimum": 1},
                        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
                        "reasoning": {"type": "string"},
                    },
                    "required": ["size_class", "count", "confidence"],
                }},
            },
            "required": ["room_type", "items"],
        },
    }
    r = client.messages.create(
        model=model or CFG["vlm_model"], max_tokens=2048,
        tools=[tool], tool_choice={"type": "tool", "name": "record_inventory"},
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": b64}},
            {"type": "text", "text": ARM_A_PROMPT},
        ]}],
    )
    for blk in r.content:
        if blk.type == "tool_use":
            return blk.input, r.usage
    return {"room_type": "unknown", "items": []}, r.usage

per_frame_A, usage_tot = [], dict(input=0, output=0)
if HAS_KEY:
    for f in frames[: CFG["vlm_max_frames"]]:
        inv, u = classify_frame(f["img"])
        inv["t"] = f["t"]
        per_frame_A.append(inv)
        usage_tot["input"] += u.input_tokens; usage_tot["output"] += u.output_tokens
        items = ", ".join(f"{i['count']}x {i['size_class']}" for i in inv["items"]) or "(none)"
        print(f"t={f['t']:5.1f}s  {inv['room_type']:8}  {items}")
    cost = usage_tot["input"] / 1e6 * 2.0 + usage_tot["output"] / 1e6 * 10.0   # Sonnet 5 list price
    print(f"\ntokens in={usage_tot['input']:,} out={usage_tot['output']:,}  ~${cost:.3f} for this room")
else:
    print("ANTHROPIC_API_KEY not set — Arm A skipped. Arm B results are still valid.")

## Stage 7 · Dedup — the same sofa in every frame

Gap **B1**, and the most common way a pipeline like this reports a wildly wrong number.
One sofa across twenty frames must count once, not twenty times.

`max` takes the highest count seen in any single frame — safe against occlusion, vulnerable
to a single false positive. `median` is the reverse. Neither is correct; proper object
tracking across frames is the real fix. This is enough to get a first reading.

In [ ]:
def dedup_counts(per_frame_invs, rule="max"):
    if not per_frame_invs: return {}
    per = []
    for inv in per_frame_invs:
        c = defaultdict(int)
        for it in inv["items"]: c[it["size_class"]] += int(it["count"])
        per.append(c)
    keys = sorted({k for c in per for k in c})
    out = {}
    for k in keys:
        series = [c.get(k, 0) for c in per]           # zeros included on purpose
        v = max(series) if rule == "max" else int(np.median(series))
        if v > 0: out[k] = v
    return out

invA = dedup_counts(per_frame_A, CFG["dedup_rule"])
# Arm B: count per class = max simultaneous detections of that class in any one frame
invB = {}
if len(B):
    per_fr = defaultdict(lambda: defaultdict(int))
    for r in B.itertuples():
        if isinstance(r.mapped_class, str): per_fr[r.t][r.mapped_class] += 1
    for cls in {c for d in per_fr.values() for c in d}:
        invB[cls] = max(d.get(cls, 0) for d in per_fr.values())

print(f"Arm A inventory ({CFG['dedup_rule']} over {len(per_frame_A)} frames)")
for k, v in sorted(invA.items()): print(f"  {v} x {k}")
print(f"\nArm B inventory (max per frame over {len(frames)} frames)")
for k, v in sorted(invB.items()): print(f"  {v} x {k}")

## Stage 8 · Aggregate — three volume figures to compare

In [ ]:
def vol_from_inventory(inv):
    return float(sum(inv[c] * CLASSES[c]["cube_m3"] for c in inv if c in CLASSES))

volA        = vol_from_inventory(invA)                 # classify -> table
volB_mapped = vol_from_inventory(invB)                 # measure -> nearest class -> table
volB_raw    = float(B["bbox_m3"].sum()) if len(B) else 0.0   # measure -> raw bounding volume

M3_TO_FT3 = 1 / 0.0283168
rows = [
    ("Arm A  classify -> cube table",    volA,        len(invA)),
    ("Arm B  measure -> class -> table",  volB_mapped, len(invB)),
    ("Arm B  measure -> raw bbox volume", volB_raw,    len(B)),
]
res = pd.DataFrame(rows, columns=["method", "volume_m3", "n_items"])
res["volume_ft3"] = res["volume_m3"] * M3_TO_FT3
print(res.round(2).to_string(index=False))
print(f"\nscale factor applied: {SCALE:.4f}  ({src})")
print("\nRaw bbox volume is expected to be LOWER than the table figures — a cube sheet")
print("carries PACKED volume, which includes padding, crating and stacking voids (gap B4).")

## Stage 9 · Evaluate — the only cell that answers the question

Needs `ground_truth.csv`. Reports **bias separately from spread**, because they have
different fixes: a system consistently 15% low is correctable with a coefficient, one
that is randomly ±15% is not.

In [ ]:
GT_PATH = ROOT / "ground_truth.csv"
if not GT_PATH.exists():
    print(f"No {GT_PATH.name} — cannot score.")
    print("Copy ground_truth_template.csv, fill it in for this room, and re-run. Gap A1.")
else:
    gt = pd.read_csv(GT_PATH)
    room = SCORE_ROOM or gt.room_id.iloc[0]   # set SCORE_ROOM in stage 0 to pick another
    print(f"available rooms: {sorted(gt.room_id.unique())}")
    g = gt[gt.room_id == room]
    true_counts = g.groupby("true_class")["qty"].sum().to_dict()
    true_vol = sum(n * CLASSES[c]["cube_m3"] for c, n in true_counts.items() if c in CLASSES)

    def score(pred, name):
        cls = set(pred) | set(true_counts)
        tp = sum(min(pred.get(c, 0), true_counts.get(c, 0)) for c in cls)
        pc, tc = sum(pred.values()), sum(true_counts.values())
        pv = vol_from_inventory(pred)
        return dict(method=name,
                    precision=round(tp / pc, 3) if pc else 0.0,
                    recall=round(tp / tc, 3) if tc else 0.0,
                    pred_items=pc, true_items=tc,
                    pred_m3=round(pv, 3), true_m3=round(true_vol, 3),
                    vol_bias_pct=round((pv - true_vol) / true_vol * 100, 1) if true_vol else np.nan,
                    vol_abs_err_pct=round(abs(pv - true_vol) / true_vol * 100, 1) if true_vol else np.nan)

    print(f"\n=== room {room} ===")
    scored = []
    if invA: scored.append(score(invA, "Arm A classify"))
    else:    print("(Arm A skipped — no API key, not scored)")
    if invB: scored.append(score(invB, "Arm B measure"))
    else:    print("(Arm B produced nothing — check stages 2 and 3)")
    if scored: print(pd.DataFrame(scored).to_string(index=False))

    # per-item dimension accuracy for Arm B
    if len(B):
        dim_rows = []
        for _, r in g.iterrows():
            m = B[B.mapped_class == r.true_class]
            if not len(m): continue
            tgt_v = r.width_m * r.depth_m * r.height_m
            m = m.loc[(m["bbox_m3"] - tgt_v).abs().idxmin()]
            truth = np.sort([r.width_m, r.depth_m, r.height_m])[::-1]
            meas  = np.sort([m.w, m.d, m.h])[::-1]
            dim_rows.append(dict(item=r.true_class,
                                 true_dims=np.round(truth, 2).tolist(),
                                 meas_dims=np.round(meas, 2).tolist(),
                                 dim_mape_pct=round(float(np.mean(np.abs(meas - truth) / truth)) * 100, 1)))
        if dim_rows:
            D = pd.DataFrame(dim_rows)
            print(f"\n--- Arm B dimension accuracy ---")
            print(D.to_string(index=False))
            print(f"\nmean dimension MAPE {D.dim_mape_pct.mean():.1f}%"
                  f"   -> implies ~{D.dim_mape_pct.mean()*3:.0f}% on volume (cube law)")

    missed = {c: n for c, n in true_counts.items() if c not in invA and c not in invB}
    if missed:
        print(f"\nMissed entirely by both arms: {missed}")

## What to do with the results

1. **Read `vol_bias_pct` before `vol_abs_err_pct`.** Consistent bias is a coefficient away
   from being fixed. Random spread is not.
2. **Re-run with `use_scale_anchor = False`** and diff stage 9. That number tells you
   whether the anchor work is worth funding, and it is the finding most likely to change
   the architecture.
3. **Compare Arm A against Arm B.** If classify-and-look-up wins — as the incumbent
   evidence suggests it will — stop building geometric measurement and put the effort into
   the size-class taxonomy and the cube table instead.
4. **Check `Missed entirely by both arms`.** Anything appearing there is a detection
   vocabulary gap, and it is cheap to fix.
5. **Then widen.** Two room types, 3–5 properties, before drawing any conclusion (gap B7).

Everything still open is tracked in `../GAPS.md`.

## Appendix · How much scale accuracy do we actually need?

Runs without any input data. Simulates measurement error and reports how often the
nearest-size-class step still picks the right class, so you know what to expect from
stage 9 before you have real footage.

Uses a **global scale error** (systematic, the way MoGe-2's error behaves) plus small
per-axis noise, then checks class assignment with and without the detector narrowing
the candidate list.

In [ ]:
import math, random

def _nearest_ranked(dims, allowed=None):
    tgt = sorted(dims, reverse=True); out = []
    for name in (allowed or CLASS_NAMES):
        c = CLASSES.get(name)
        if not c: continue
        cand = sorted(c["typical_dims_m"], reverse=True)
        nrm = math.sqrt(sum(x*x for x in cand)) or 1e-6
        out.append((math.sqrt(sum((a-b)**2 for a, b in zip(tgt, cand)))/nrm, name))
    out.sort(); return out

# --- which size classes are genuinely hard to separate on geometry alone? ---
print("Tightest discriminations (best vs runner-up, detector-restricted):\n")
margins = []
for prompt, cands in VOCAB.items():
    if len(cands) < 2: continue
    for truth in cands:
        r = _nearest_ranked(CLASSES[truth]["typical_dims_m"], cands)
        margins.append((r[1][0] - r[0][0], truth, r[1][1]))
for m, a, b in sorted(margins)[:6]:
    print(f"  {a:22} vs {b:22} margin {m:.3f}")

# --- how does class accuracy hold up as scale error grows? ---
random.seed(7)
GROUPS = [k for k, v in VOCAB.items() if len(v) > 1]
print(f"\nClass-assignment accuracy, 2000 trials per row, {len(GROUPS)} ambiguous groups:\n")
print(f"  {'scale err':>10} {'restricted':>12} {'unrestricted':>14}")
for se in [0.00, 0.05, 0.08, 0.15]:
    okr = oku = n = 0
    for _ in range(2000):
        cands = VOCAB[random.choice(GROUPS)]
        truth = random.choice(cands)
        sc = 1.0 + random.gauss(0, se)
        dims = [d * sc * (1 + random.gauss(0, 0.03)) for d in CLASSES[truth]["typical_dims_m"]]
        okr += _nearest_ranked(dims, cands)[0][1] == truth
        oku += _nearest_ranked(dims)[0][1] == truth
        n += 1
    flag = "  <- MoGe-2 published" if se == 0.08 else ""
    if se == 0.08: gap8 = (okr - oku) / n * 100
    print(f"  {se*100:9.0f}% {okr/n*100:11.1f}% {oku/n*100:13.1f}%{flag}")

print(f"""
Two things to take from this:

1. Narrowing candidates by the detector label is load-bearing. At MoGe-2's published 8%
   scale error it is worth about {gap8:.0f} percentage points of class accuracy.
   Never map a measurement against the whole cube table.

2. Class assignment degrades gracefully with scale error, but the tight pairs go first.
   Note which pairs appear at the top of the margin list above — those are the ones to
   check by hand in stage 9, and the ones where a size question to the customer buys
   more than better geometry.
""")